In [ ]:
!python --version
!nvidia-smi --query-gpu=name,driver_version --format=csv

Python 3.13.15
name, driver_version
Tesla T4, 580.82.07


In [ ]:
!pip uninstall -y -q torch torchvision torchaudio
!pip install -q paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr pymupdf pillow tqdm scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 553.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 34.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 831.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!python -c "import paddle; paddle.utils.run_check()"

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Running verify PaddlePaddle program ... 
/usr/local/lib/python3.13/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(
I1012 11:07:16.185220  3690 pir_interpreter.cc:1524] New Executor is Running ...
W1012 11:07:16.186947  3690 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 12.6
I1012 11:07:16.189865  3690 pir_interpreter.cc:1547] pir interpreter is running by multi-thread mode ...
PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep lea

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/doc_classifier")
DS = ROOT / "dataset"

rows = []
for f in DS.rglob("*"):
    if f.is_file() and len(f.relative_to(DS).parts) == 3:
        split, label, name = f.relative_to(DS).parts
        rows.append({"path": str(f), "split": split, "label": label,
                     "source": name.split("_")[0].lower(), "ext": f.suffix.lower()})
df = pd.DataFrame(rows)
print(len(df), "files")
print(df.groupby(["split", "label"]).size().unstack())
print(df[df.split == "test"].ext.value_counts())

442 files
label  bill  invoice  receipt
split                        
test     21       30       30
train    91      130      140
ext
.pdf    81
Name: count, dtype: int64


In [ ]:
import io
import time
import numpy as np
import pymupdf          # opens PDFs
import paddle
from PIL import Image, ImageOps
from paddleocr import PaddleOCR

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
print("running on:", paddle.device.get_device())   # we want gpu:0

ocr = PaddleOCR(
    lang="en",
    use_doc_orientation_classify=False,   # skip extra features we don't need
    use_doc_unwarping=False,
)

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.


running on: gpu:0


Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


In [ ]:
def read_image(img):
    # fix phones' rotation flag and make sure the image has 3 colour channels
    img = ImageOps.exif_transpose(img).convert("RGB")

    # Paddle wants a numpy array in BGR order, so we flip the colour channels
    pixels = np.array(img)[:, :, ::-1]

    result = ocr.predict(pixels)
    if not result:
        return ""
    lines = result[0]["rec_texts"]      # list of text lines Paddle found
    return "\n".join(lines)

In [ ]:
def extract_text(path, max_pages=3):
    path = str(path)
    try:
        if path.lower().endswith(".pdf"):
            pdf = pymupdf.open(path)
            all_pages = []
            for page in list(pdf)[:max_pages]:
                # turn the page into a picture, even for digital PDFs,
                # so every file goes through OCR the same way
                picture = page.get_pixmap(dpi=200)
                img = Image.open(io.BytesIO(picture.tobytes("png")))
                all_pages.append(read_image(img))
            pdf.close()
            return "\n".join(all_pages), "ocr_pdf"
        else:
            return read_image(Image.open(path)), "ocr_image"
    except Exception as e:
        return "", f"error: {e}"

In [ ]:
test_files = [
    df[df.path.str.endswith("cord_0002.jpg")].iloc[0].path,   # Tesseract got 0 chars here
    df[df.path.str.endswith("cord_0000.jpg")].iloc[0].path,
    df[(df.source == "rvl") & (df.split == "train")].iloc[0].path,
    df[(df.source == "fatura") & (df.split == "test")].iloc[0].path,
    df[(df.label == "bill") & (df.split == "test")].iloc[0].path,
]
for p in test_files:
    print(Path(p).name)

cord_0002.jpg
cord_0000.jpg
rvl_0000.jpg
fatura_0000_t31.pdf
realbill_0000.pdf


In [ ]:
for p in test_files:
    start = time.time()
    text, method = extract_text(p)
    seconds = time.time() - start

    print(Path(p).name, "|", method, "|", len(text.strip()), "chars |", round(seconds, 1), "s")
    print(text[:150].replace("\n", " "))
    print("-" * 40)

cord_0002.jpg | ocr_image | 108 chars | 3.0 s
1066-Choconut Bun 28.000 x1 28.000 6001-Plastic Bag Small 0x1 0 Total Item: 28.000 Total. Cash 40.000 12.000
----------------------------------------
cord_0000.jpg | ocr_image | 156 chars | 1.1 s
15.000 Mineral Water (S) 20.000 Pocky Chocolate 42.000 Nerds Strw Grape Nerds Trop Punch 42.000 42.000 Nerds Watermelon 161.000 TOTAL 161.000 CASH CHA
----------------------------------------
rvl_0000.jpg | ocr_image | 954 chars | 2.1 s
APR 8 '94 6:54 FROM AGWAY-WAUERLY PAGE.802 WAY AGWAY, INC., COUNTRY FOODS DIVISION INVOICE PO BOX 3285, SYRACUSE, NY 13221-3285 3 W) (4/88) MER NO. CR
----------------------------------------
fatura_0000_t31.pdf | ocr_pdf | 532 chars | 2.5 s
White and Sons Email:cpetersen@example.com www.WhiteandSons.org COMMERCIAL INVOICE Bill to:Angela Barnes MD INVOICE # INV19819578 8858 Wells Tunnel Su
----------------------------------------
realbill_0000.pdf | ocr_pdf | 426 chars | 2.3 s
Electricity Bills How to read your elec

In [ ]:
SAVE_FILE = ROOT / "ocr_results.csv"

if SAVE_FILE.exists():
    results = pd.read_csv(SAVE_FILE).to_dict("records")
    print("resuming, already done:", len(results))
else:
    results = []
    print("starting fresh")

already_done = set(r["path"] for r in results)
todo = df[~df.path.isin(already_done)]
print("files left to read:", len(todo))

resuming, already done: 442
files left to read: 0


In [ ]:
from tqdm import tqdm

count = 0
for row in tqdm(todo.to_dict("records")):
    text, method = extract_text(row["path"])
    row["text"] = text
    row["method"] = method
    results.append(row)

    count += 1
    if count % 25 == 0:                       # save a checkpoint every 25 files
        pd.DataFrame(results).to_csv(SAVE_FILE, index=False)

pd.DataFrame(results).to_csv(SAVE_FILE, index=False)
print("done")

In [ ]:
data = pd.DataFrame(results)
data["text"] = data["text"].fillna("")
data["n_chars"] = data["text"].str.strip().str.len()

print(data.method.value_counts())
print()
print("median characters per source:")
print(data.groupby("source").n_chars.median())
print()
print("files with under 30 characters:", (data.n_chars < 30).sum())

method
ocr_image    310
ocr_pdf      132
Name: count, dtype: int64

median characters per source:
source
cord         119.0
fatura       575.0
katanaml     938.5
realbill    1858.5
rvl          751.0
sroie        635.0
synth        867.5
Name: n_chars, dtype: float64

files with under 30 characters: 0


# Performing EDA

In [ ]:
import re

def clean_text(text):
    text = text.lower()                         # all lowercase
    text = re.sub(r"\d+", "0", text)            # every number becomes 0 (words matter, not digits)
    text = re.sub(r"[^a-z0 ]+", " ", text)      # remove symbols and OCR junk
    text = re.sub(r"\s+", " ", text).strip()    # collapse extra spaces
    return text

data["clean"] = data["text"].apply(clean_text)
print(data[["source", "clean"]].sample(3, random_state=1))

    source                                              clean
246   cord  receipt 0 plastic bag 0 0 0 pink panther p 0 0...
425    rvl  0 hazleton laboratories america inc telephone ...
293  sroie  0 ai 0 epgtle 0tel0 gst0 0n0 ar 0 klang selang...


In [ ]:
dupes = data[data.duplicated("clean", keep=False)]
print("rows with duplicate text:", len(dupes))
print(dupes[["split", "label", "source", "path"]].sort_values("path"))

rows with duplicate text: 2
     split    label source                                               path
263  train  receipt  sroie  /content/drive/MyDrive/doc_classifier/dataset/...
287  train  receipt  sroie  /content/drive/MyDrive/doc_classifier/dataset/...


In [ ]:
print([Path(p).name for p in dupes.path])
data = data.drop_duplicates("clean", keep="first").reset_index(drop=True)
print("rows now:", len(data))

['sroie_0005.jpg', 'sroie_0044.jpg']
rows now: 441


Checking whcih files doens't have their own keyword

In [ ]:
keywords = {
    "invoice": r"invoice|bill to|due date|order|qty",
    "receipt": r"total|cash|change|subtotal|tax",
    "bill":    r"bill|amount due|account|usage|meter|due",
}

def has_own_keywords(row):
    return bool(re.search(keywords[row["label"]], row["clean"]))

data["keyword_ok"] = data.apply(has_own_keywords, axis=1)
flagged = data[~data.keyword_ok]

print("files with none of their own class's keywords:", len(flagged))
print(flagged.groupby(["label", "source"]).size())

files with none of their own class's keywords: 30
label    source
invoice  fatura     2
         rvl       27
receipt  cord       1
dtype: int64


In [ ]:
rvl_flagged = flagged[flagged.source == "rvl"]

print("how much text they have:")
print(rvl_flagged.n_chars.describe())

how much text they have:
count      27.000000
mean      638.925926
std       428.643650
min        40.000000
25%       296.500000
50%       594.000000
75%       810.500000
max      1798.000000
Name: n_chars, dtype: float64


# Saving the clean dataset into my drive

In [ ]:
CLEAN_FILE = ROOT / "clean_dataset.csv"
data.to_csv(CLEAN_FILE, index=False)

print("saved", len(data), "rows")
print(data.groupby(["split", "label"]).size().unstack())

saved 441 rows
label  bill  invoice  receipt
split                        
test     21       30       30
train    91      130      139


# Initiating the model training from here...

In [ ]:
train = data[data.split == "train"]
test = data[data.split == "test"]

X_train_text = train["clean"]
y_train = train["label"]
X_test_text = test["clean"]
y_test = test["label"]

print("train documents:", len(train))
print("test documents: ", len(test))

train documents: 360
test documents:  81


Converting the text into numbers using TF_IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),     # single words and pairs like "grand total"
    min_df=2,               # ignore words that appear in only one document
    sublinear_tf=True,      # stops very frequent words from dominating
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

print("train matrix:", X_train.shape)
print("test matrix: ", X_test.shape)

train matrix: (360, 4238)
test matrix:  (81, 4238)


Choosing which words matter most for each class

In [ ]:
import numpy as np

words = np.array(tfidf.get_feature_names_out())

for label in sorted(y_train.unique()):
    rows = (y_train == label).values
    average = np.asarray(X_train[rows].mean(axis=0)).ravel()
    top = average.argsort()[::-1][:12]
    print(label.upper(), "->", ", ".join(words[top]))
    print()

BILL -> bill, payment, at, charge, pay, this, period, www, com, number, reading, rs

INVOICE -> invoice, us, example, email, date, net, price, total, to, worth, each, site

RECEIPT -> total, cash, total cash, gst, change, rm, sr, cash change, no, tax, subtotal, item



Training and comparing on both Logistic Regression and Random Forest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
forest = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)

In [ ]:
log_reg.fit(X_train, y_train)
forest.fit(X_train, y_train)
print("both models trained")

both models trained


Testing on the unseen *data*

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def show_results(name, model):
    predictions = model.predict(X_test)
    print("=" * 50)
    print(name)
    print("accuracy:", round(accuracy_score(y_test, predictions), 3))
    print("macro F1:", round(f1_score(y_test, predictions, average="macro"), 3))
    print(classification_report(y_test, predictions))
    return predictions

log_pred = show_results("Logistic Regression", log_reg)
forest_pred = show_results("Random Forest", forest)

Logistic Regression
accuracy: 1.0
macro F1: 1.0
              precision    recall  f1-score   support

        bill       1.00      1.00      1.00        21
     invoice       1.00      1.00      1.00        30
     receipt       1.00      1.00      1.00        30

    accuracy                           1.00        81
   macro avg       1.00      1.00      1.00        81
weighted avg       1.00      1.00      1.00        81

Random Forest
accuracy: 0.988
macro F1: 0.986
              precision    recall  f1-score   support

        bill       1.00      0.95      0.98        21
     invoice       0.97      1.00      0.98        30
     receipt       1.00      1.00      1.00        30

    accuracy                           0.99        81
   macro avg       0.99      0.98      0.99        81
weighted avg       0.99      0.99      0.99        81



Checking if the test data has similar file type like training

In [ ]:
fatura = data[data.source == "fatura"].copy()
fatura["template"] = fatura.path.str.extract(r"_t(\d+)")[0]

train_templates = set(fatura[fatura.split == "train"].template)
test_templates = set(fatura[fatura.split == "test"].template)

print("templates in train:", len(train_templates))
print("templates in test: ", len(test_templates))
print("test templates also seen in training:", len(test_templates & train_templates))

templates in train: 30
templates in test:  10
test templates also seen in training: 0


In [ ]:
print("train layouts:", m_train.layout.value_counts().to_dict())
print("test layouts: ", m_test.layout.value_counts().to_dict())
print()
print("train bill types:", m_train.bill_type.value_counts().to_dict())
print("test bill types: ", m_test.bill_type.value_counts().to_dict())
print()
print("test formats:", m_test.format.value_counts().to_dict())

train layouts: {'layout_stub': 47, 'layout_banner': 43}
test layouts:  {'layout_mono': 20}

train bill types: {'electricity': 18, 'water': 18, 'gas': 18, 'internet': 18, 'mobile': 18}
test bill types:  {'electricity': 4, 'water': 4, 'gas': 4, 'internet': 4, 'mobile': 4}

test formats: {'pdf': 11, 'scanned_pdf': 4, 'jpg': 3, 'png': 2}


In [ ]:
def leave_source_out(make_model):
    scores = {}
    for source in ["rvl", "fatura", "katanaml", "cord", "sroie"]:
        train_rows = data[(data.split == "train") & (data.source != source)]
        eval_rows = data[data.source == source]

        model = make_model()
        model.fit(train_rows.clean, train_rows.label)

        predictions = model.predict(eval_rows.clean)
        scores[source] = (predictions == eval_rows.label).mean()
    return scores

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

candidates = {
    "baseline: words + LogReg": lambda: make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=1000, class_weight="balanced")),

    "words, drop very common (max_df=0.5)": lambda: make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, sublinear_tf=True),
        LogisticRegression(max_iter=1000, class_weight="balanced")),

    "character n-grams + LogReg": lambda: make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=1000, class_weight="balanced")),

    "words + LogReg, stronger regularization": lambda: make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=1000, class_weight="balanced", C=0.1)),

    "words + Linear SVM": lambda: make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LinearSVC(class_weight="balanced")),

    "words + Random Forest": lambda: make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)),
}

table = {}
for name, make_model in candidates.items():
    scores = leave_source_out(make_model)
    scores["AVERAGE"] = sum(scores.values()) / len(scores)
    table[name] = scores
    print(name, "->", round(scores["AVERAGE"], 2))

print()
print(pd.DataFrame(table).T.round(2))

baseline: words + LogReg -> 0.48
words, drop very common (max_df=0.5) -> 0.44
character n-grams + LogReg -> 0.58
words + LogReg, stronger regularization -> 0.53
words + Linear SVM -> 0.45
words + Random Forest -> 0.32

                                          rvl  fatura  katanaml  cord  sroie  \
baseline: words + LogReg                 0.15    0.62       1.0  0.30   0.32   
words, drop very common (max_df=0.5)     0.08    0.70       1.0  0.21   0.19   
character n-grams + LogReg               0.40    0.99       1.0  0.28   0.22   
words + LogReg, stronger regularization  0.20    0.61       1.0  0.33   0.48   
words + Linear SVM                       0.13    0.60       1.0  0.32   0.20   
words + Random Forest                    0.02    0.36       0.8  0.00   0.43   

                                         AVERAGE  
baseline: words + LogReg                    0.48  
words, drop very common (max_df=0.5)        0.44  
character n-grams + LogReg                  0.58  
words + LogReg, 

#Saving the model

In [ ]:
import joblib

joblib.dump(tfidf, ROOT / "tfidf.joblib")
joblib.dump(log_reg, ROOT / "model.joblib")
print("saved to", ROOT)

saved to /content/drive/MyDrive/doc_classifier


In [ ]:
import numpy as np

confidence = log_reg.predict_proba(X_test).max(axis=1)
print("lowest:", confidence.min().round(2))
print("5th percentile:", np.percentile(confidence, 5).round(2))
print("median:", np.median(confidence).round(2))

lowest: 0.41
5th percentile: 0.71
median: 0.85
